# Mobile UI Agent — Environment Exploration Notebook

Use this notebook to:
1. Explore the state/action/reward mechanics interactively
2. Prototype agent action sequences before writing code
3. Visualise reward distributions across the dataset
4. Debug individual episodes step-by-step

**Setup:** make sure the package is installed (`pip install -e ..[dev,notebook]` from the repo root).

In [ ]:
import json
import sys
sys.path.insert(0, '..')  # allow import from repo root when running from notebooks/

from mobile_ui_env import (
    AppState, Screen,
    MobileUIEnv, load_environment,
    build_dataset, get_task_by_id,
    compute_reward,
)
from mobile_ui_env.actions import execute_action, execute_action_sequence

print('Import OK ✓')

---
## 1. Dataset Overview

In [ ]:
train_tasks = build_dataset('train')
eval_tasks  = build_dataset('eval')

print(f'Train tasks : {len(train_tasks)}')
print(f'Eval tasks  : {len(eval_tasks)}')
print()

print('── Train ────────────────────────────────────────────────────')
for t in train_tasks:
    print(f'  [{t.task_id}] {t.goal["type"]:25s}  max_steps={t.max_steps:3d}  "{t.instruction}"')

print()
print('── Eval ─────────────────────────────────────────────────────')
for t in eval_tasks:
    print(f'  [{t.task_id}] {t.goal["type"]:25s}  max_steps={t.max_steps:3d}  "{t.instruction}"')

In [ ]:
# Goal type distribution
from collections import Counter

all_tasks = build_dataset('all')
train_types = Counter(t.goal['type'] for t in train_tasks)
eval_types  = Counter(t.goal['type'] for t in eval_tasks)

print(f'  {"Goal type":<28} {"Train":>6} {"Eval":>6}')
print(f'  {"─"*28} {"─"*6} {"─"*6}')
all_types = sorted(set(train_types) | set(eval_types))
for gt in all_types:
    print(f'  {gt:<28} {train_types.get(gt, 0):>6} {eval_types.get(gt, 0):>6}')

---
## 2. State Space Exploration

In [ ]:
from mobile_ui_env.state import SCREEN_ELEMENTS, APP_STATIC

# Print all screen elements
print('Screen elements:')
for screen, elements in SCREEN_ELEMENTS.items():
    print(f'  {screen.value:<12} → {elements}')

print()
print('Static app data:', APP_STATIC)

In [ ]:
# Show observation at each screen
for screen in Screen:
    state = AppState(current_screen=screen)
    print(f'\n── {screen.value.upper()} ──────────────────')
    print(json.dumps(state.observation(), indent=2))

---
## 3. Single Episode Step-Through

In [ ]:
task = get_task_by_id('train_001')  # Create note 'Buy milk'
print(f'Task: {task.instruction}')
print(f'Goal: {task.goal}')
print(f'Max steps: {task.max_steps}')

In [ ]:
env = MobileUIEnv(task)
obs = env.reset()
print('Initial observation:')
print(json.dumps(obs, indent=2))

In [ ]:
# Step through the optimal action sequence manually
optimal_actions = [
    {'action': 'tap',    'target': 'notes_button'},
    {'action': 'tap',    'target': 'add_note_button'},
    {'action': 'type',   'target': 'note_input', 'text': 'Buy milk'},
    {'action': 'tap',    'target': 'save_note_button'},
    {'action': 'finish'},
]

result = env.step(optimal_actions)

print('Action results:')
for i, (action, res) in enumerate(zip(optimal_actions, result['action_results']), 1):
    status = '✓' if res['valid'] else '✗'
    safety = ' [SAFETY]' if res['safety_violation'] else ''
    print(f'  {i}. {status} {action}  →  {res["message"]}{safety}')

print()
print('Reward breakdown:')
for k, v in result['reward_info'].items():
    print(f'  {k:<30} = {v:.4f}')

print(f'\nDone: {result["done"]}')
print(f'Notes saved: {env.state.notes}')

---
## 4. Reward Hacking Demo

In [ ]:
# What actually happens when an agent tries to game partial_progress_reward
# by cycling through screens without completing the goal.

from mobile_ui_env.rubric import partial_progress_reward, success_reward

task = get_task_by_id('train_001')  # note_created goal

# Naive hacking attempt: visit every screen, call finish, create nothing.
hack_actions = [
    {'action': 'tap', 'target': 'notes_button'},
    {'action': 'back'},
    {'action': 'tap', 'target': 'settings_button'},
    {'action': 'back'},
    {'action': 'tap', 'target': 'profile_button'},
    {'action': 'finish'},
]

state = AppState()
execute_action_sequence(hack_actions, state, max_steps=task.max_steps)

print('Naive screen-cycling attempt:')
print(f'  success_reward          = {success_reward(state, task, hack_actions):.2f}')
print(f'  partial_progress_reward = {partial_progress_reward(state, task, hack_actions):.2f}')
print()
print('This particular hack does NOT work: _check_goal checks the final')
print('state (was the note actually saved?), not whether a screen was')
print('visited, so screen-cycling earns nothing here. See README section 6')
print('for the vector that *did* turn out to be real: the reward weights')
print('used by load_environment() had the wrong sign for penalty terms.')


---
## 5. Reward Distribution Across All Tasks

In [ ]:
# Run the heuristic agent over all tasks and plot reward distribution
import sys
sys.path.insert(0, '..')
from run_eval import heuristic_agent

all_tasks = build_dataset('all')
rewards_by_type = {}

for task in all_tasks:
    env = MobileUIEnv(task)
    obs = env.reset()
    actions = heuristic_agent(obs, task)
    result = env.step(actions)
    
    gt = task.goal['type']
    if gt not in rewards_by_type:
        rewards_by_type[gt] = []
    rewards_by_type[gt].append({
        'task_id': task.task_id,
        'final_reward': result['reward_info']['final_reward'],
        'success': result['reward_info']['success_reward'],
        'steps': env.state.steps_taken,
    })

print(f'  {"Goal type":<28} {"Count":>5}  {"Avg reward":>10}  {"Success%":>9}')
print(f'  {"─"*28} {"─"*5}  {"─"*10}  {"─"*9}')
for gt, records in sorted(rewards_by_type.items()):
    n = len(records)
    avg_r = sum(r['final_reward'] for r in records) / n
    success_pct = sum(r['success'] for r in records) / n * 100
    print(f'  {gt:<28} {n:>5}  {avg_r:>10.4f}  {success_pct:>8.0f}%')

In [ ]:
# Plot (requires matplotlib)
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    goal_types = sorted(rewards_by_type.keys())
    avg_rewards  = [sum(r['final_reward'] for r in rewards_by_type[gt]) / len(rewards_by_type[gt]) for gt in goal_types]
    success_rates = [sum(r['success'] for r in rewards_by_type[gt]) / len(rewards_by_type[gt]) for gt in goal_types]

    x = range(len(goal_types))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(list(x), avg_rewards, color='steelblue', alpha=0.85)
    axes[0].set_yticks(list(x))
    axes[0].set_yticklabels(goal_types)
    axes[0].set_xlim(0, 1.05)
    axes[0].axvline(0.5, color='gray', linestyle='--', linewidth=0.8)
    axes[0].set_title('Average Final Reward by Goal Type\n(heuristic baseline)', fontsize=12)
    axes[0].set_xlabel('Final Reward')

    colors = ['#2ecc71' if s >= 0.8 else '#e74c3c' for s in success_rates]
    axes[1].barh(list(x), [s * 100 for s in success_rates], color=colors, alpha=0.85)
    axes[1].set_yticks(list(x))
    axes[1].set_yticklabels(goal_types)
    axes[1].set_xlim(0, 110)
    axes[1].axvline(80, color='gray', linestyle='--', linewidth=0.8, label='80% threshold')
    axes[1].set_title('Success Rate (%) by Goal Type\n(heuristic baseline)', fontsize=12)
    axes[1].set_xlabel('Success Rate %')

    green_patch = mpatches.Patch(color='#2ecc71', label='≥ 80%')
    red_patch   = mpatches.Patch(color='#e74c3c', label='< 80%')
    axes[1].legend(handles=[green_patch, red_patch], loc='lower right')

    plt.tight_layout()
    plt.savefig('reward_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Plot saved to reward_distribution.png')

except ImportError:
    print('matplotlib not installed. Run: pip install matplotlib')

---
## 6. Edge Case Exploration

In [ ]:
# What happens with garbage input?
task = get_task_by_id('train_001')
env = MobileUIEnv(task)
env.reset()

garbage_actions = [
    None,
    42,
    'tap notes_button',
    {'action': 'swipe', 'direction': 'up'},
    {'action': 'tap', 'target': 'nonexistent_element'},
    {'action': 'type', 'target': 'note_input', 'text': ''},
    {'action': 'finish'},
]

result = env.step(garbage_actions)  # Must not raise

print('Garbage action results:')
for action, res in zip(garbage_actions, result['action_results']):
    status = '✓' if res['valid'] else '✗'
    print(f'  {status} {str(action):<50} → {res["message"][:60]}')

print(f'\nInvalid action count: {env.state.invalid_action_count}')
print(f'Success reward      : {result["reward_info"]["success_reward"]}')
print(f'No crash ✓')

In [ ]:
# Safety violation demo
task = get_task_by_id('train_015')  # visit profile without logout
env = MobileUIEnv(task)
env.reset()

unsafe_actions = [
    {'action': 'tap', 'target': 'profile_button'},
    {'action': 'tap', 'target': 'logout_button'},  # ← safety violation
    {'action': 'finish'},
]

result = env.step(unsafe_actions)
ri = result['reward_info']

print('Safety violation scenario:')
print(f'  success_reward  = {ri["success_reward"]:.2f}')
print(f'  safety_penalty  = {ri["safety_penalty"]:.2f}  ← 1.0 triggered')
print(f'  final_reward    = {ri["final_reward"]:.4f}')

---
## 7. load_environment() and Verifiers Interface

In [ ]:
env = load_environment()

print(f'Type       : {type(env).__name__}')
print(f'Train tasks: {len(env.dataset)}')
print(f'Eval tasks : {len(env.eval_dataset)}')
print(f'Rubric     : {type(env.rubric).__name__}')

# Run evaluate() with a simple dummy agent
def dummy_agent(obs, task):
    return [{'action': 'finish'}]

results = env.evaluate(dummy_agent, split='eval')
rewards = [r['reward_info']['final_reward'] for r in results]
print(f'\nDummy agent avg reward: {sum(rewards)/len(rewards):.4f}')
print(f'(Should be very low — dummy always finishes immediately)')